In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric = pd.read_csv("metric.csv")
trace = pd.read_csv("trace.csv")
log = pd.read_csv("log.csv")
error_logs = pd.read_csv("error_logs.csv")

# Parse timestamps to datetime (UTC) as required, keep original integer seconds too
for df in (metric, trace, log, error_logs):
    if 'timestamp' in df.columns:
        df['timestamp_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Helper to build a compact one-row summary DataFrame and top-name counts
def build_summary(df, name_col=None, top_n=20, message_mode=False):
    total_rows = int(df.shape[0])
    cmdb_ids = sorted(df['cmdb_id'].dropna().unique().tolist()) if 'cmdb_id' in df.columns else []
    earliest_ts = int(df['timestamp'].min()) if (('timestamp' in df.columns) and (not df['timestamp'].empty)) else None
    latest_ts = int(df['timestamp'].max()) if (('timestamp' in df.columns) and (not df['timestamp'].empty)) else None
    summary_df = pd.DataFrame([{
        'total_rows': total_rows,
        'cmdb_ids': cmdb_ids,
        'earliest_ts': earliest_ts,
        'latest_ts': latest_ts
    }])
    # top names or messages
    if name_col and name_col in df.columns:
        top_df = df[name_col].value_counts().reset_index(name='count').rename(columns={'index': name_col}).head(top_n)
    elif message_mode and 'message' in df.columns:
        top_df = df['message'].value_counts().reset_index(name='count').rename(columns={'index': 'message'}).head(top_n)
    else:
        top_df = pd.DataFrame(columns=['name','count'])
    return summary_df, top_df

# Build summaries for each file
metric_summary_df, metric_top_kpis = build_summary(metric, name_col='kpi_name', top_n=20)
trace_summary_df, trace_top = build_summary(trace, name_col='trace_name', top_n=20)
log_summary_df, log_top = build_summary(log, name_col='log_name', top_n=20)
error_summary_df, error_top = build_summary(error_logs, message_mode=True, top_n=20)

# Display compact outputs (one summary per file + top items)
metric_summary_df, metric_top_kpis, trace_summary_df, trace_top, log_summary_df, log_top, error_summary_df, error_top

```
Out[1]:
```
summary = (
    "Telemetry load summary:\n"
    "- metric.csv: 8792 rows; cmdb_ids include many ts-* services; earliest_ts=1706144040, latest_ts=1706145480. "
    "Top KPIs (by count): cpu=1700, mem=1700, socket=1700, diskio=1675, workload=700, latency-50=525, latency-90=525, error=267.\n"
    "- trace.csv: 9968 rows; cmdb_ids include root and many ts-* services; earliest_ts=1706144040, latest_ts=1706145480. "
    "Top trace features (by count): trace.from_root.* entries (275 each), trace.to_ts-station-service.* (125 each), "
    "trace.from_ts-food-service.* (100 each), trace.to_ts-travel-service.* (100 each), trace.from_ts-travel2-service.* (100 each).\n"
    "- log.csv: 1902 rows; cmdb_ids include ts-* services; earliest_ts=1706144040, latest_ts=1706145480. "
    "Top log names: log.error_count=951, log.row_count=951.\n"
    "- error_logs.csv: 0 rows (no error log messages present).\n"
    "Overall: metric/trace/log telemetry are populated across many services with consistent time range; no raw error log messages were found."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(   total_rows                                           cmdb_ids  earliest_ts   latest_ts
0        8792  [ts-admin-basic-info-service, ts-admin-order-s...   1706144040  1706145480,      kpi_name  count
0         cpu   1700
1         mem   1700
2      socket   1700
3      diskio   1675
4    workload    700
5  latency-50    525
6  latency-90    525
7       error    267,    total_rows                                           cmdb_ids  earliest_ts   latest_ts
0        9968  [root, ts-admin-basic-info-service, ts-admin-t...   1706144040  1706145480,                                      trace_name  count
0                     trace.from_root.row_count    275
1                    trace.from_root.error_rate    275
2                  trace.from_root.duration_p95    275
3                 trace.from_root.duration_mean    275
4         trace.to_ts-station-service.row_count    125
5        trace.to_ts-station-service.error_rate    125
6      trace.to_ts-station-service.duration_p95    125
7     trace.to_ts-station-service.duration_mean    125
8          trace.from_ts-food-service.row_count    100
9         trace.from_ts-food-service.error_rate    100
10      trace.from_ts-food-service.duration_p95    100
11     trace.from_ts-food-service.duration_mean    100
12     trace.to_ts-travel-service.duration_mean    100
13      trace.to_ts-travel-service.duration_p95    100
14        trace.to_ts-travel-service.error_rate    100
15         trace.to_ts-travel-service.row_count    100
16  trace.from_ts-travel2-service.duration_mean    100
17   trace.from_ts-travel2-service.duration_p95    100
18     trace.from_ts-travel2-service.error_rate    100
19      trace.from_ts-travel2-service.row_count    100,    total_rows                                           cmdb_ids  earliest_ts   latest_ts
0        1902  [ts-admin-basic-info-service, ts-admin-travel-...   1706144040  1706145480,           log_name  count
0  log.error_count    951
1    log.row_count    951,    total_rows cmdb_ids earliest_ts latest_ts
0           0       []        None      None, Empty DataFrame
Columns: [message, count]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd

# reuse existing metric if available, else load
try:
    metric
except NameError:
    metric = pd.read_csv("metric.csv")

# keep full series in memory (metric variable). Restrict to target cmdb_ids
targets = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]
metric_sub = metric[metric['cmdb_id'].isin(targets)].copy()

# Ensure timestamp and value types
metric_sub['timestamp'] = metric_sub['timestamp'].astype(int)
metric_sub['value'] = metric_sub['value'].astype(float)

# Compute global statistics per (cmdb_id, kpi_name) using the full series for these cmdb_ids
global_stats = metric_sub.groupby(['cmdb_id','kpi_name'])['value'].agg(
    global_count='count',
    global_mean='mean',
    global_p95=lambda s: s.quantile(0.95),
    global_p5=lambda s: s.quantile(0.05)
).reset_index()

# Parse timestamp to UTC datetimes (keep original integer seconds)
metric_sub['timestamp_dt'] = pd.to_datetime(metric_sub['timestamp'], unit='s', utc=True)

# Define incident window (UTC) inclusive
window_start = pd.Timestamp("2024-01-25 00:51:17", tz='UTC')
window_end   = pd.Timestamp("2024-01-25 01:21:17", tz='UTC')

# Filter rows to incident window (inclusive)
window_df = metric_sub[(metric_sub['timestamp_dt'] >= window_start) & (metric_sub['timestamp_dt'] <= window_end)].copy()

# If no data in window, return empty DataFrame with desired columns
if window_df.empty:
    result_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','count_in_window','earliest_ts_in_window','latest_ts_in_window',
        'count_above_p95_in_window','max_value_in_window','count_below_p5_in_window','min_value_in_window',
        'global_p95','global_p5'
    ])
else:
    # Attach global stats to each row so we can count above/below thresholds
    window_with_globals = window_df.merge(global_stats[['cmdb_id','kpi_name','global_p95','global_p5']],
                                         on=['cmdb_id','kpi_name'], how='left')

    # Flags for comparisons against global thresholds
    window_with_globals['above_p95'] = window_with_globals['value'] > window_with_globals['global_p95']
    window_with_globals['below_p5'] = window_with_globals['value'] < window_with_globals['global_p5']

    # Aggregate per (cmdb_id, kpi_name)
    agg = window_with_globals.groupby(['cmdb_id','kpi_name']).agg(
        count_in_window=('value','size'),
        earliest_ts_in_window=('timestamp','min'),
        latest_ts_in_window=('timestamp','max'),
        count_above_p95_in_window=('above_p95','sum'),
        max_value_in_window=('value','max'),
        count_below_p5_in_window=('below_p5','sum'),
        min_value_in_window=('value','min'),
        global_p95=('global_p95','first'),
        global_p5=('global_p5','first')
    ).reset_index()

    # Ensure integer timestamps
    agg['earliest_ts_in_window'] = agg['earliest_ts_in_window'].astype(int)
    agg['latest_ts_in_window'] = agg['latest_ts_in_window'].astype(int)

    # Filter rows with count_in_window > 0 (should already be true) and sort
    result_df = agg[agg['count_in_window'] > 0].sort_values(
        by=['count_above_p95_in_window','count_in_window'], ascending=[False, False]
    ).reset_index(drop=True).head(50)

# Display the compact result table
result_df

```
Out[2]:
```
Summary of findings (plain English):

- Data coverage: 36 distinct (service, KPI) series from the five target services had data inside the incident window. Most series contain 25 samples in the window (earliest_ts_in_window=1706144040, latest_ts_in_window=1706145480).

- Brief, small spikes above historical 95th percentile:
  - Many KPIs (cpu, diskio, latency-50, latency-90, mem, socket, workload, etc.) for multiple services show a small number of samples above their global p95 during the window. The top entries each have 2 samples above global_p95 (so very short spikes), and a few series have 1 or 0 samples above p95.
  - Examples:
    - ts-auth-service: cpu had 25 samples in-window, 2 samples above its global_p95 (max ~16.87 vs global_p95 ~16.72).
    - ts-auth-service: diskio had 25 samples, 2 above p95 (max ~67,423 vs global_p95 ~57,861).
    - ts-auth-service: latency-90 had 25 samples, 2 above p95 (max ~0.9057 s vs global_p95 ~0.8749 s).
    - ts-travel-service: mem had 25 samples, 2 above p95 (max ~524,249,000 vs global_p95 ~524,232,300).
    - ts-order-service: socket had 25 samples, 1 above p95.
  - The "error" KPI for ts-travel-service remains 0 (no errors) in-window.

- Interpretation: There are short-lived spikes in several KPIs across the target services during the incident window, but they are infrequent (1–2 samples each). No KPI shows sustained or widespread exceedance of global p95 in this window.

If you want, I can (a) show the full compact table of all 36 rows, (b) list only series with count_above_p95_in_window > 0, or (c) convert the integer timestamps to human-readable UTC times. Which would you prefer?

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  count_in_window  earliest_ts_in_window  latest_ts_in_window  count_above_p95_in_window  max_value_in_window  count_below_p5_in_window  min_value_in_window    global_p95     global_p5
0     ts-auth-service         cpu               25             1706144040           1706145480                          2         1.687222e+01                         2         1.343522e+01  1.672293e+01  1.385532e+01
1     ts-auth-service      diskio               25             1706144040           1706145480                          2         6.742312e+04                         2         3.752794e+04  5.786105e+04  4.035667e+04
2     ts-auth-service  latency-50               25             1706144040           1706145480                          2         2.672857e-01                         2         2.021526e-01  2.576778e-01  2.039772e-01
3     ts-auth-service  latency-90               25             1706144040           1706145480                          2         9.057066e-01                         2         4.593947e-01  8.749454e-01  4.858270e-01
4     ts-auth-service         mem               25             1706144040           1706145480                          2         2.416256e+08                         2         2.399044e+08  2.415560e+08  2.402256e+08
..                ...         ...              ...                    ...                  ...                        ...                  ...                       ...                  ...           ...           ...
31  ts-travel-service         mem               25             1706144040           1706145480                          2         5.242490e+08                         2         2.623707e+08  5.242323e+08  2.632720e+08
32  ts-travel-service      socket               25             1706144040           1706145480                          2         1.777778e+01                         2         1.153333e+01  1.733667e+01  1.177000e+01
33  ts-travel-service    workload               25             1706144040           1706145480                          2         2.134317e+00                         2         1.375533e+00  2.078883e+00  1.458887e+00
34   ts-order-service      socket               25             1706144040           1706145480                          1         1.158333e+01                         2         9.000000e+00  1.100000e+01  9.183333e+00
35  ts-travel-service       error               25             1706144040           1706145480                          0         0.000000e+00                         0         0.000000e+00  0.000000e+00  0.000000e+00

[36 rows x 11 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd

# reuse existing 'trace' variable if present, else load
try:
    trace
except NameError:
    trace = pd.read_csv("trace.csv")

# Ensure types
trace['timestamp'] = trace['timestamp'].astype(int)
trace['value'] = trace['value'].astype(float)

# Compute global thresholds (p95 and p5) per (cmdb_id, trace_name) using full series BEFORE filtering
global_trace_stats = trace.groupby(['cmdb_id','trace_name'])['value'].agg(
    global_p95=lambda s: s.quantile(0.95),
    global_p5=lambda s: s.quantile(0.05)
).reset_index()

# Parse timestamps to UTC datetimes (keep original integer seconds)
trace['timestamp_dt'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)

# Define incident window (UTC) inclusive
window_start = pd.Timestamp("2024-01-25 00:51:17", tz='UTC')
window_end   = pd.Timestamp("2024-01-25 01:21:17", tz='UTC')

# Filter rows to the incident window (inclusive)
trace_window = trace[(trace['timestamp_dt'] >= window_start) & (trace['timestamp_dt'] <= window_end)].copy()

# If no data in window, prepare empty result with required columns
if trace_window.empty:
    result_df = pd.DataFrame(columns=[
        'cmdb_id','trace_name','count_in_window','earliest_ts_in_window','latest_ts_in_window',
        'count_above_global_p95_in_window','max_value_in_window','global_p95'
    ])
else:
    # Attach global p95 to window rows
    tw = trace_window.merge(global_trace_stats, on=['cmdb_id','trace_name'], how='left')

    # Flag above global p95
    tw['above_global_p95'] = tw['value'] > tw['global_p95']

    # Aggregate per (cmdb_id, trace_name)
    agg = tw.groupby(['cmdb_id','trace_name']).agg(
        count_in_window=('value','size'),
        earliest_ts_in_window=('timestamp','min'),
        latest_ts_in_window=('timestamp','max'),
        count_above_global_p95_in_window=('above_global_p95','sum'),
        max_value_in_window=('value','max'),
        global_p95=('global_p95','first')
    ).reset_index()

    # Ensure integer timestamps
    agg['earliest_ts_in_window'] = agg['earliest_ts_in_window'].astype(int)
    agg['latest_ts_in_window'] = agg['latest_ts_in_window'].astype(int)

    # Select rows with count_above_global_p95_in_window > 0
    positives = agg[agg['count_above_global_p95_in_window'] > 0].copy()

    if positives.empty:
        # fallback: top 20 by count_in_window
        result_df = agg.sort_values(by=['count_in_window','earliest_ts_in_window'], ascending=[False, True]).head(20)
    else:
        # sort positives as requested: by count_above desc then earliest_ts asc, limit 20
        result_df = positives.sort_values(by=['count_above_global_p95_in_window','earliest_ts_in_window'],
                                          ascending=[False, True]).head(20)

    # Keep only requested columns and their order
    result_df = result_df[[
        'cmdb_id','trace_name','count_in_window','earliest_ts_in_window','latest_ts_in_window',
        'count_above_global_p95_in_window','max_value_in_window','global_p95'
    ]].reset_index(drop=True)

# Display the compact result
result_df

```
Out[3]:
```
Summary (plain English):

- What was done: global p95/p5 were computed from the full trace series before filtering; then traces were filtered to the incident window 2024-01-25 00:51:17 — 2024-01-25 01:21:17 UTC.

- Key result: multiple trace series (all from cmdb_id "root") had short spikes above their global p95 during the window. The top 20 trace series each have 25 samples in-window and show 2 samples exceeding the global p95.

- Example affected trace targets (each had 2 samples > global p95):
  - trace.to_ts-admin-basic-info-service.duration_* (duration_mean and duration_p95)
  - trace.to_ts-admin-travel-service.duration_mean and duration_p95
  - trace.to_ts-assurance-service.duration_mean, duration_p95, row_count
  - trace.to_ts-auth-service.duration_mean and duration_p95
  - trace.to_ts-consign-service.duration_mean and duration_p95
  - trace.to_ts-food-service.duration_mean, duration_p95, row_count
  - trace.to_ts-inside-payment-service.duration_mean and duration_p95
  - trace.to_ts-order-other-service.duration_mean and duration_p95
  - trace.to_ts-order-service.duration_mean and duration_p95

- Interpretation: The deviations are brief (2 samples per series out of 25), indicating short-lived latency increases on multiple downstream calls from the root service rather than sustained, widespread trace-level degradation.

If you want, I can (a) show the full compact table of all trace series that exceeded p95, (b) convert the integer timestamps to human-readable UTC times, or (c) correlate these trace spikes with the metric KPI spikes. Which would you like next?

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                         trace_name  count_in_window  earliest_ts_in_window  latest_ts_in_window  count_above_global_p95_in_window  max_value_in_window  global_p95
0     root  trace.to_ts-admin-basic-info-service.duration_...               25             1706144040           1706145480                                 2             0.012272    0.012080
1     root  trace.to_ts-admin-basic-info-service.duration_p95               25             1706144040           1706145480                                 2             0.020204    0.017499
2     root     trace.to_ts-admin-travel-service.duration_mean               25             1706144040           1706145480                                 2             0.256368    0.253767
3     root      trace.to_ts-admin-travel-service.duration_p95               25             1706144040           1706145480                                 2             0.459142    0.443115
4     root        trace.to_ts-assurance-service.duration_mean               25             1706144040           1706145480                                 2             0.004293    0.004152
5     root         trace.to_ts-assurance-service.duration_p95               25             1706144040           1706145480                                 2             0.007970    0.007100
6     root            trace.to_ts-assurance-service.row_count               25             1706144040           1706145480                                 2            14.000000   12.600000
7     root             trace.to_ts-auth-service.duration_mean               25             1706144040           1706145480                                 2             0.387383    0.366411
8     root              trace.to_ts-auth-service.duration_p95               25             1706144040           1706145480                                 2             1.267147    1.115602
9     root          trace.to_ts-consign-service.duration_mean               25             1706144040           1706145480                                 2             0.132057    0.030023
10    root           trace.to_ts-consign-service.duration_p95               25             1706144040           1706145480                                 2             0.450441    0.057100
11    root             trace.to_ts-food-service.duration_mean               25             1706144040           1706145480                                 2             0.063466    0.060673
12    root              trace.to_ts-food-service.duration_p95               25             1706144040           1706145480                                 2             0.135654    0.125719
13    root                 trace.to_ts-food-service.row_count               25             1706144040           1706145480                                 2            29.000000   27.200000
14    root   trace.to_ts-inside-payment-service.duration_mean               25             1706144040           1706145480                                 2             0.017969    0.014537
15    root    trace.to_ts-inside-payment-service.duration_p95               25             1706144040           1706145480                                 2             0.046409    0.020269
16    root      trace.to_ts-order-other-service.duration_mean               25             1706144040           1706145480                                 2             0.053549    0.051794
17    root       trace.to_ts-order-other-service.duration_p95               25             1706144040           1706145480                                 2             0.059846    0.058682
18    root            trace.to_ts-order-service.duration_mean               25             1706144040           1706145480                                 2             0.088555    0.087516
19    root             trace.to_ts-order-service.duration_p95               25             1706144040           1706145480                                 2             0.122932    0.114502```
```

In [ ]:
# In[4]:

import pandas as pd

# reuse existing 'log' variable if present, else load
try:
    log
except NameError:
    log = pd.read_csv("log.csv")

# Ensure types
log['timestamp'] = log['timestamp'].astype(int)
log['value'] = log['value'].astype(float)

# Compute global thresholds (p95 and p5) per (cmdb_id, log_name) using full series BEFORE filtering
global_log_stats = log.groupby(['cmdb_id','log_name'])['value'].agg(
    global_p95=lambda s: s.quantile(0.95),
    global_p5=lambda s: s.quantile(0.05)
).reset_index()

# Parse timestamps to UTC datetimes (keep original integer seconds)
log['timestamp_dt'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)

# Define incident window (UTC) inclusive
window_start = pd.Timestamp("2024-01-25 00:51:17", tz='UTC')
window_end   = pd.Timestamp("2024-01-25 01:21:17", tz='UTC')

# Filter rows to the incident window (inclusive)
log_window = log[(log['timestamp_dt'] >= window_start) & (log['timestamp_dt'] <= window_end)].copy()

# Prepare empty result structure if no data in window
columns = [
    'cmdb_id','log_name','count_in_window','earliest_ts_in_window','latest_ts_in_window',
    'count_above_global_p95_in_window','max_value_in_window',
    'count_below_global_p5_in_window','min_value_in_window','global_p95','global_p5'
]
if log_window.empty:
    result_log = pd.DataFrame(columns=columns)
else:
    # Merge global thresholds into window rows
    lw = log_window.merge(global_log_stats, on=['cmdb_id','log_name'], how='left')

    # Flags for comparisons against global thresholds
    lw['above_global_p95'] = lw['value'] > lw['global_p95']
    lw['below_global_p5'] = lw['value'] < lw['global_p5']

    # Aggregate per (cmdb_id, log_name)
    agg = lw.groupby(['cmdb_id','log_name']).agg(
        count_in_window=('value','size'),
        earliest_ts_in_window=('timestamp','min'),
        latest_ts_in_window=('timestamp','max'),
        count_above_global_p95_in_window=('above_global_p95','sum'),
        max_value_in_window=('value','max'),
        count_below_global_p5_in_window=('below_global_p5','sum'),
        min_value_in_window=('value','min'),
        global_p95=('global_p95','first'),
        global_p5=('global_p5','first')
    ).reset_index()

    # Ensure integer timestamps
    agg['earliest_ts_in_window'] = agg['earliest_ts_in_window'].astype(int)
    agg['latest_ts_in_window'] = agg['latest_ts_in_window'].astype(int)

    # Filter rows where count_above>0 OR count_below>0
    flagged = agg[(agg['count_above_global_p95_in_window'] > 0) | (agg['count_below_global_p5_in_window'] > 0)].copy()

    if flagged.empty:
        # If none flagged, return top 20 by count_in_window (still limited to requested columns)
        result_log = agg.sort_values(by=['count_in_window','earliest_ts_in_window'], ascending=[False, True]).head(20)
    else:
        # Sort by count_above desc then count_below desc, limit 20
        result_log = flagged.sort_values(by=['count_above_global_p95_in_window','count_below_global_p5_in_window'],
                                         ascending=[False, False]).head(20)

    # Keep only requested columns and their order
    result_log = result_log[columns].reset_index(drop=True)

# Display compact result
result_log

```
Out[4]:
```
Summary (plain English):

- What I did: computed global p95/p5 per (cmdb_id, log_name) from the full log series, then looked at the incident window 2024-01-25 00:51:17 – 2024-01-25 01:21:17 UTC and reported series that showed deviations vs their global thresholds.

- Key findings (top flagged log series):
  - ts-order-other-service (log.row_count): 25 samples in-window, 2 samples above global p95 (max=42.0, global_p95=38.6) and 2 below global p5 (min=4.0, global_p5=5.8).
  - ts-order-service (log.row_count): 25 samples, 2 above p95 (max=75.0 vs p95=59.8) and 2 below p5 (min=8.0 vs p5=13.0).
  - ts-route-service (log.row_count): 25 samples, 2 above p95 (max=387.0 vs p95=373.0) and 2 below p5 (min=94.0 vs p5=247.0).
  - ts-travel2-service (log.row_count): 25 samples, 2 above p95 (max=561.0 vs p95=546.2) and 2 below p5 (min=127.0 vs p5=373.4).
  - ts-ui-dashboard, ts-assurance-service, ts-food-map-service, ts-food-service, ts-auth-service, ts-basic-service, ts-consign-service, ts-inside-payment-service, ts-travel-service and several mongo-related services also show 1–2 samples above their global p95 (and some also have samples below p5).
  - Most flagged series have earliest_ts_in_window=1706144040 and latest_ts_in_window=1706145480 (i.e., same window bounds).

- Interpretation: multiple services experienced brief deviations in log row counts during the incident window — several show short spikes above their historical 95th percentile (indicating transient increased logging/traffic), and many also show samples below their historical 5th percentile (indicating transient drops). The largest absolute spikes appear in route/travel-related and some auth/order services (e.g., max row_count 387, 561, 380). Mongo/backing-store series show small absolute counts but still triggered threshold crossings.

If you want, I can (a) show only the series with above-p95 counts, (b) convert the integer timestamps to human-readable UTC times, or (c) correlate these log spikes with metric/trace anomalies. Which next?

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id       log_name  count_in_window  earliest_ts_in_window  latest_ts_in_window  count_above_global_p95_in_window  max_value_in_window  count_below_global_p5_in_window  min_value_in_window  global_p95  global_p5
0        ts-order-other-service  log.row_count               25             1706144040           1706145480                                 2                 42.0                                2                  4.0        38.6        5.8
1              ts-order-service  log.row_count               25             1706144040           1706145480                                 2                 75.0                                2                  8.0        59.8       13.0
2              ts-route-service  log.row_count               25             1706144040           1706145480                                 2                387.0                                2                 94.0       373.0      247.0
3            ts-travel2-service  log.row_count               25             1706144040           1706145480                                 2                561.0                                2                127.0       546.2      373.4
4               ts-ui-dashboard  log.row_count               25             1706144040           1706145480                                 2                296.0                                2                 75.0       291.6      207.2
5          ts-assurance-service  log.row_count               25             1706144040           1706145480                                 2                 14.0                                1                  2.0        12.6        3.0
6           ts-food-map-service  log.row_count               25             1706144040           1706145480                                 2                 29.0                                1                  4.0        27.2       13.0
7               ts-food-service  log.row_count               25             1706144040           1706145480                                 2                 87.0                                1                 12.0        81.6       39.0
8   ts-admin-basic-info-service  log.row_count               25             1706144040           1706145480                                 1                 32.0                                2                 11.0        31.0       18.4
9               ts-auth-service  log.row_count               25             1706144040           1706145480                                 1                380.0                                2                 92.0       374.0      264.0
10             ts-basic-service  log.row_count               25             1706144040           1706145480                                 1                230.0                                2                 56.0       224.0      158.0
11           ts-consign-service  log.row_count               25             1706144040           1706145480                                 1                204.0                                2                 14.0        84.0       15.4
12    ts-inside-payment-service  log.row_count               25             1706144040           1706145480                                 1                 24.0                                2                  4.0        22.0        4.4
13            ts-travel-service  log.row_count               25             1706144040           1706145480                                 1                538.0                                2                148.0       535.0      337.4
14           ts-assurance-mongo  log.row_count               24             1706144040           1706145420                                 1                  2.0                                0                  1.0         1.0        1.0
15                ts-auth-mongo  log.row_count               24             1706144100           1706145480                                 1                  8.0                                0                  1.0         1.0        1.0
16              ts-config-mongo  log.row_count               24             1706144100           1706145480                                 1                  2.0                                0                  1.0         1.0        1.0
17             ts-consign-mongo  log.row_count               24             1706144040           1706145420                                 1                  4.0                                0                  1.0         1.0        1.0
18            ts-contacts-mongo  log.row_count               24             1706144100           1706145480                                 1                  2.0                                0                  1.0         1.0        1.0
19                ts-food-mongo  log.row_count               24             1706144040           1706145420                                 1                  2.0                                0                  1.0         1.0        1.0```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# reuse existing 'trace' and 'global_trace_stats' if present, else load/compute
try:
    trace
except NameError:
    trace = pd.read_csv("trace.csv")

# Ensure correct dtypes
trace['timestamp'] = trace['timestamp'].astype(int)
trace['value'] = trace['value'].astype(float)

# Compute global_p95 per (cmdb_id, trace_name) using full series BEFORE filtering (if not already computed)
try:
    global_trace_stats
except NameError:
    global_trace_stats = trace.groupby(['cmdb_id','trace_name'])['value'].agg(
        global_p95=lambda s: s.quantile(0.95),
        global_p5=lambda s: s.quantile(0.05)
    ).reset_index()

# Parse timestamps to UTC datetimes (keep integer seconds)
trace['timestamp_dt'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)

# Incident window (UTC) inclusive
window_start = pd.Timestamp("2024-01-25 00:51:17", tz='UTC')
window_end   = pd.Timestamp("2024-01-25 01:21:17", tz='UTC')

# Filter to window
trace_window = trace[(trace['timestamp_dt'] >= window_start) & (trace['timestamp_dt'] <= window_end)].copy()

# Candidate services and pattern matching for to_ts- or from_ts- occurrences
candidates = ['ts-auth-service','ts-order-service','ts-route-service','ts-train-service','ts-travel-service']
pattern = r'(to_ts|from_ts)-(' + '|'.join([s.replace('-', r'\-') for s in candidates]) + r')'

# Keep only rows whose trace_name mentions any candidate service (to_ts- or from_ts-)
trace_window_candidates = trace_window[trace_window['trace_name'].str.contains(pattern, regex=True, na=False)].copy()

# Merge global_p95 (computed using full series) on (cmdb_id, trace_name)
tw = trace_window_candidates.merge(
    global_trace_stats[['cmdb_id','trace_name','global_p95']],
    on=['cmdb_id','trace_name'],
    how='left'
)

if tw.empty:
    result = pd.DataFrame(columns=[
        'trace_name','count_in_window','count_above_global_p95_in_window',
        'earliest_anomaly_ts_in_window','max_value_in_window','global_p95'
    ])
else:
    # Flag rows above global p95
    tw['above_global_p95'] = tw['value'] > tw['global_p95']

    # Aggregate per trace_name (across any cmdb_id)
    def earliest_anomaly(series_ts, series_flag):
        # series_ts: timestamps (int), series_flag: boolean above p95
        if series_flag.any():
            return int(series_ts[series_flag].min())
        else:
            return np.nan

    agg = tw.groupby('trace_name').apply(
        lambda df: pd.Series({
            'count_in_window': int(df.shape[0]),
            'count_above_global_p95_in_window': int(df['above_global_p95'].sum()),
            'earliest_anomaly_ts_in_window': (int(df.loc[df['above_global_p95'], 'timestamp'].min())
                                              if df['above_global_p95'].any() else np.nan),
            'max_value_in_window': float(df['value'].max()),
            'global_p95': float(df['global_p95'].iloc[0])  # same global_p95 for this trace_name/cmdb_id combo
        })
    ).reset_index()

    # Keep only rows with anomalies (count_above_global_p95_in_window > 0)
    anomalies = agg[agg['count_above_global_p95_in_window'] > 0].copy()

    # Sort by earliest_anomaly_ts ascending and limit to top 20
    result = anomalies.sort_values(by='earliest_anomaly_ts_in_window', ascending=True).head(20).reset_index(drop=True)

# Display requested columns in required order
result = result[[
    'trace_name','count_in_window','count_above_global_p95_in_window',
    'earliest_anomaly_ts_in_window','max_value_in_window','global_p95'
]]
result.head(20)

```
Out[5]:
```
summary = (
    "Summary: After computing global p95 per trace on the full trace series and filtering traces to the incident window "
    "2024-01-25 00:51:17–2024-01-25 01:21:17 UTC, then selecting trace_name entries that reference any of the candidate services "
    "ts-auth-service, ts-order-service, ts-route-service, ts-train-service, or ts-travel-service (matching both to_ts- and from_ts-), "
    "no trace series had any samples above their global p95 in the incident window. The result set is empty."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [trace_name, count_in_window, count_above_global_p95_in_window, earliest_anomaly_ts_in_window, max_value_in_window, global_p95]
Index: []```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np

# Reuse loaded variables if present, else load
try:
    metric
except NameError:
    metric = pd.read_csv("metric.csv")
try:
    log
except NameError:
    log = pd.read_csv("log.csv")

# Ensure types
metric['timestamp'] = metric['timestamp'].astype(int)
metric['value'] = metric['value'].astype(float)
log['timestamp'] = log['timestamp'].astype(int)
log['value'] = log['value'].astype(float)

# Target component and incident window (UTC)
component = 'ts-route-service'
window_start = pd.Timestamp("2024-01-25 00:51:17", tz='UTC')
window_end   = pd.Timestamp("2024-01-25 01:21:17", tz='UTC')

# 1) Compute global thresholds (p95, p5) for metric KPIs of ts-route-service using the full metric.csv
metric_route = metric[metric['cmdb_id'] == component].copy()
global_metric_stats = metric_route.groupby('kpi_name')['value'].agg(
    global_p95=lambda s: s.quantile(0.95),
    global_p5=lambda s: s.quantile(0.05)
).reset_index()

# 2) Compute global thresholds (p95, p5) for log names of ts-route-service using the full log.csv
log_route = log[log['cmdb_id'] == component].copy()
global_log_stats = log_route.groupby('log_name')['value'].agg(
    global_p95=lambda s: s.quantile(0.95),
    global_p5=lambda s: s.quantile(0.05)
).reset_index()

# 3) Parse timestamps to UTC datetimes for filtering (keep original integer timestamps)
metric_route['timestamp_dt'] = pd.to_datetime(metric_route['timestamp'], unit='s', utc=True)
log_route['timestamp_dt'] = pd.to_datetime(log_route['timestamp'], unit='s', utc=True)

# 4) Filter both datasets to the incident window (inclusive)
metric_window = metric_route[(metric_route['timestamp_dt'] >= window_start) & (metric_route['timestamp_dt'] <= window_end)].copy()
log_window = log_route[(log_route['timestamp_dt'] >= window_start) & (log_route['timestamp_dt'] <= window_end)].copy()

# Helper to aggregate series (generic for metric kpi_name or log log_name)
def aggregate_series(df, series_col, global_stats_df):
    if df.empty:
        return pd.DataFrame(columns=[
            'component','series_name','count_in_window','earliest_ts_in_window','latest_ts_in_window',
            'count_above_global_p95_in_window','earliest_ts_above_global_p95_in_window','max_value_in_window',
            'count_below_global_p5_in_window','earliest_ts_below_global_p5_in_window','min_value_in_window',
            'global_p95','global_p5'
        ])
    # Merge global thresholds
    merged = df.merge(global_stats_df, left_on=series_col, right_on=global_stats_df.columns[0], how='left')
    # Flag anomalies
    merged['above_p95'] = merged['value'] > merged['global_p95']
    merged['below_p5'] = merged['value'] < merged['global_p5']
    # Aggregate
    agg = merged.groupby(series_col).apply(lambda g: pd.Series({
        'component': component,
        'series_name': g.name,
        'count_in_window': int(g.shape[0]),
        'earliest_ts_in_window': int(g['timestamp'].min()),
        'latest_ts_in_window': int(g['timestamp'].max()),
        'count_above_global_p95_in_window': int(g['above_p95'].sum()),
        'earliest_ts_above_global_p95_in_window': (int(g.loc[g['above_p95'], 'timestamp'].min()) 
                                                   if g['above_p95'].any() else pd.NA),
        'max_value_in_window': float(g['value'].max()),
        'count_below_global_p5_in_window': int(g['below_p5'].sum()),
        'earliest_ts_below_global_p5_in_window': (int(g.loc[g['below_p5'], 'timestamp'].min()) 
                                                  if g['below_p5'].any() else pd.NA),
        'min_value_in_window': float(g['value'].min()),
        'global_p95': float(g['global_p95'].iloc[0]),
        'global_p5': float(g['global_p5'].iloc[0])
    })).reset_index(drop=True)
    # Ensure nullable integer dtype for earliest anomaly timestamps
    for col in ['earliest_ts_above_global_p95_in_window','earliest_ts_below_global_p5_in_window']:
        agg[col] = agg[col].astype('Int64')
    return agg

# Aggregate metric KPIs
metric_agg = aggregate_series(metric_window, 'kpi_name', global_metric_stats)

# Aggregate log series
log_agg = aggregate_series(log_window, 'log_name', global_log_stats)

# Combine results
combined = pd.concat([metric_agg, log_agg], ignore_index=True)

# Filter rows where count_above>0 OR count_below>0
flagged = combined[(combined['count_above_global_p95_in_window'] > 0) | (combined['count_below_global_p5_in_window'] > 0)].copy()

# Sort by earliest_ts_below_global_p5_in_window ascending (nulls last) then earliest_ts_above_global_p95_in_window ascending
flagged.sort_values(
    by=['earliest_ts_below_global_p5_in_window','earliest_ts_above_global_p95_in_window'],
    ascending=[True, True],
    inplace=True,
    na_position='last'
)

# Limit to top 20 rows
result_df = flagged.head(20).reset_index(drop=True)

# Keep requested column order
result_df = result_df[[
    'component','series_name','count_in_window','earliest_ts_in_window','latest_ts_in_window',
    'count_above_global_p95_in_window','earliest_ts_above_global_p95_in_window','max_value_in_window',
    'count_below_global_p5_in_window','earliest_ts_below_global_p5_in_window','min_value_in_window',
    'global_p95','global_p5'
]]

# Display compact result
result_df

```
Out[6]:
```
Summary (plain English):

- What I did: computed global p95/p5 for ts-route-service from the full metric and log series, then inspected the incident window 2024-01-25 00:51:17 — 2024-01-25 01:21:17 UTC.

- Key finding: ts-route-service shows both brief spikes above its historical 95th percentile and brief dips below its historical 5th percentile across multiple metric KPIs and its log.row_count. Only series with anomalies are reported.

- Affected series (each has 25 samples in-window, 2 samples > global_p95 and 2 samples < global_p5):
  - log.row_count — max 387.0, global_p95=373.0; earliest below at 2024-01-25 00:54:00 UTC, earliest above at 2024-01-25 00:56:00 UTC.
  - mem — max 256,413,700; global_p95=256,297,700; earliest below at 2024-01-25 00:54:00 UTC, earliest above at 2024-01-25 01:17:00 UTC.
  - diskio — max 71,902.71; global_p95=56,861.50; earliest below at 2024-01-25 00:54:00 UTC, earliest above at 2024-01-25 00:57:00 UTC.
  - workload — max 12.53912; global_p95=12.26976; earliest below at 2024-01-25 00:57:00 UTC, earliest above at 2024-01-25 00:59:00 UTC.
  - cpu — max 1.741613; global_p95=1.729241; earliest below at 2024-01-25 01:05:00 UTC, earliest above at 2024-01-25 01:09:00 UTC.
  - socket — max 14.5; global_p95=14.41; earliest below at 2024-01-25 01:06:00 UTC, earliest above at 2024-01-25 00:55:00 UTC.
  - latency-50 — max 0.003548038; global_p95=0.003532136; earliest below at 2024-01-25 01:13:00 UTC, earliest above at 2024-01-25 00:56:00 UTC.
  - latency-90 — max 0.007901956; global_p95=0.007781613; earliest below at 2024-01-25 01:13:00 UTC, earliest above at 2024-01-25 00:56:00 UTC.

- Interpretation: ts-route-service experienced short-lived, concurrent anomalies — both upward spikes and downward drops — across its log row count and several resource/latency KPIs within the incident window. These deviations are transient (only 1–2 samples each) and occur at different times between ~00:54 and ~01:18 UTC.

The original code execution output of IPython Kernel is also provided below for reference:

component    series_name  count_in_window  earliest_ts_in_window  latest_ts_in_window  count_above_global_p95_in_window  earliest_ts_above_global_p95_in_window  max_value_in_window  count_below_global_p5_in_window  earliest_ts_below_global_p5_in_window  min_value_in_window    global_p95     global_p5
0  ts-route-service  log.row_count               25             1706144040           1706145480                                 2                              1706144160         3.870000e+02                                2                             1706144040         9.400000e+01  3.730000e+02  2.470000e+02
1  ts-route-service            mem               25             1706144040           1706145480                                 2                              1706145420         2.564137e+08                                2                             1706144040         2.553749e+08  2.562977e+08  2.554607e+08
2  ts-route-service         diskio               25             1706144040           1706145480                                 2                              1706144220         7.190271e+04                                2                             1706144100         4.408285e+04  5.686150e+04  4.640675e+04
3  ts-route-service       workload               25             1706144040           1706145480                                 2                              1706144340         1.253912e+01                                2                             1706144220         9.933383e+00  1.226976e+01  1.035129e+01
4  ts-route-service            cpu               25             1706144040           1706145480                                 2                              1706144940         1.741613e+00                                2                             1706144700         1.504613e+00  1.729241e+00  1.513320e+00
5  ts-route-service         socket               25             1706144040           1706145480                                 2                              1706144100         1.450000e+01                                2                             1706144760         1.300000e+01  1.441000e+01  1.300667e+01
6  ts-route-service     latency-50               25             1706144040           1706145480                                 2                              1706144160         3.548038e-03                                2                             1706145180         3.268878e-03  3.532136e-03  3.280883e-03
7  ts-route-service     latency-90               25             1706144040           1706145480                                 2                              1706144160         7.901956e-03                                2                             1706145180         5.828703e-03  7.781613e-03  5.989399e-03```
```